In [17]:
!pip install torcheval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 2.1 MB/s eta 0:00:00a 0:00:01


In [ ]:
!pip install transformers

In [1]:
from torchvision.transforms import v2
from torchvision import tv_tensors


from pathlib import Path
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import kagglehub
import datasets
import torch
from torchvision import transforms
import math
from datasets import load_dataset
from torcheval.metrics.functional import multiclass_f1_score
# Download latest version
path = kagglehub.competition_download('airbus-ship-detection')

print("Path to competition files:", path)


/home/fakeegor/my_projects/my_ASR_project/tuning_proj/training_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: /home/fakeegor/.cache/kagglehub/competitions/airbus-ship-detection


In [2]:
from transformers import default_data_collator

In [3]:
path_obj = Path(path)
train_segmentations = pd.read_csv(path_obj / "train_ship_segmentations_v2.csv")
obj = train_segmentations[~train_segmentations.EncodedPixels.isna()].iloc[1]
image = cv2.imread(path_obj / "train_v2" / obj.ImageId)

In [4]:
positive_part = train_segmentations.EncodedPixels.index[train_segmentations.EncodedPixels.notna()]
negative_part = train_segmentations.EncodedPixels.index[~train_segmentations.EncodedPixels.notna()]
np.random.choice(positive_part, size = (int(positive_part.shape[0]*0.8))),np.asarray(train_segmentations.index)
# training_paths = np.random.shuffle(np.concatenate((np.random.choice(positive_part, size = (positive_part.shape[0]*0.8)),
# np.random.choice(negative_part, size = (negative_part.shape[0]*0.8)))))


(array([211593,  31398, 191058, ..., 184263,  12961,  95863],
       shape=(65378,)),
 array([     0,      1,      2, ..., 231720, 231721, 231722],
       shape=(231723,)))

In [5]:
np.random.uniform(0, 1)

0.5802659517197163

In [8]:
class BackBoneDataset(datasets.GeneratorBasedBuilder):
    def __init__(self, datasetPaths, requiresMaskedLabels, 
                turn_prob = 0.5,
                prob_configs = {"hor_flip":0.4,
                                 "vertical_flip":0.3,
                                 "gaussian_blur":0.5},
                image_shape = {
                    "h":768,
                    "w":768,
                    "c":3
                }):
        
        self.datasetPaths = datasetPaths ## add random permutation
        self.requiresMaskedLabels = requiresMaskedLabels
        self.probabilities = prob_configs
        self.HWChannels = image_shape
        self.turnProb = turn_prob
        super().__init__()
        
        # create selecting positive and negative examples from the dataset
        positive_part = self.datasetPaths.EncodedPixels.index[self.datasetPaths.EncodedPixels.notna()]
        negative_part = self.datasetPaths.EncodedPixels.index[~self.datasetPaths.EncodedPixels.notna()]
        self.training_paths = np.concatenate((np.random.choice(positive_part, size = int(positive_part.shape[0]*0.8)),
        np.random.choice(negative_part, size = int(negative_part.shape[0]*0.8))))
        np.random.shuffle(self.training_paths)

        
        self.testing_paths = np.array( list(set(np.asarray(self.datasetPaths.index)).difference(set(self.training_paths))))


    def _info(self):
        return datasets.DatasetInfo(
            description="training dataset for cv task",
            features = datasets.Features({
            "path": datasets.Value("string"),
            "images": datasets.Array3D(shape=(self.HWChannels["c"],
                                             self.HWChannels["h"],
                                             self.HWChannels["w"]),
                                    dtype="float32"),
            "labels": datasets.Value("int8") if not self.requiresMaskedLabels else \
                                    datasets.Array3D(shape=(1,
                                             self.HWChannels["h"],
                                             self.HWChannels["w"]),
                                    dtype="float32")
            }))

    def _split_generators(self, dl_manager):

        return [
            datasets.SplitGenerator(name=datasets.Split.TRAIN,
                                    gen_kwargs = {
                                        "split": datasets.Split.TRAIN, 
                                        "indices": self.training_paths,
                                        "imageId_list":self.datasetPaths.loc[self.training_paths].ImageId,
                                        "EncodedPixels_list": self.datasetPaths.loc[self.training_paths].EncodedPixels
                                    }),
            datasets.SplitGenerator(name=datasets.Split.TEST,
                                    gen_kwargs = {
                                        "split": datasets.Split.TEST,
                                        "indices": self.testing_paths,
                                        "imageId_list":self.datasetPaths.loc[self.testing_paths].ImageId,
                                        "EncodedPixels_list": self.datasetPaths.loc[self.testing_paths].EncodedPixels
                                    }
                                    )
        ]
        

    def _generate_examples(self, split, indices, imageId_list, EncodedPixels_list):
        # yield and random seed
        # self.paths = random permute

        # create callback on epoch end(if streaming) -> after processing a certain amount of steps, to randomize training examples
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        #random.seed(worker_seed)
        def create_mask_with_gaussian(image, coords):
            image_ = np.zeros_like(image[:, :, 0]).astype(np.uint8)
            
            # coords_ = coords[coords[:, 0].argsort()]
            # coords_ = coords_[coords_[:, 1].argsort()]
            kernels = np.random.randint(5, 15)*np.ones(2).astype(int)*2+1
            image_ = cv2.fillPoly(image_, [coords], color=(255, 255, 255))
            
            image_ = cv2.GaussianBlur(image_.astype(float), ksize=kernels, sigmaX=kernels[0], sigmaY=kernels[1])
            return image_

        def bounding_box(item_):
            coords_ = item_.split(" ")
            #coords_ = np.array(coords_span[::2]).astype(int)
            #coords_.sort()
            item = [(int(c)//image.shape[1], int(c)%image.shape[1]) for c in coords_[::2]]
            spans = [int(sp) for sp in coords_[1::2]]
            item = item + [(item[i][0], item[i][1]+spans[i]) for i in range(len(item))]
            
            item = np.asarray(item)
            y = item[item[:, 1].argmin()].reshape(-1, 2)
            y_min = y[y[:, 0].argmin()]

            x = item[item[:, 0].argmin()].reshape(-1, 2)
            x_min = x[x[:, 1].argmax()]

            x = item[item[:, 1].argmax()].reshape(-1, 2)
            x_max = x[x[:, 0].argmin()]

            y = item[item[:, 0].argmax()].reshape(-1, 2)
            y_max = y[y[:, 1].argmax()]
            return np.array((y_min, x_min, x_max, y_max))
        for i in np.arange(imageId_list.shape[0]):
            print()
            label = EncodedPixels_list.iloc[i]
            image = cv2.imread(path_obj / "train_v2" / imageId_list.iloc[i])
            bounding_box_ = bounding_box(label) if not pd.isna(label) else None

            if self.requiresMaskedLabels:
                target = create_mask_with_gaussian(image, bounding_box_)
            else:
                target = int(not pd.isna(label))
            if split == 'train':
                if bounding_box_ is not None:
                    mass = np.mean(bounding_box_, axis=0)
                    if self.turnProb >= np.random.uniform(0, 1):
                        distances = np.asarray([[image.shape[0]//2,image.shape[1]], [image.shape[0]//2, 0], [0, image.shape[1]//2], [image.shape[0], image.shape[1]//2]])
                        #min_distance = np.linalg.norm(bounding_box_[:, :, None] - np.ones((4, *bounding_box_.shape))*bounding_box_, axis=-1)
                        degr = (-5+ np.random.random()*10)/57.29
                    
                        min_ = np.argmin(np.linalg.norm(np.transpose(np.ones((4, *bounding_box_.shape))*bounding_box_, axes = (1, 0, 2)) - np.ones((4, *bounding_box_.shape))*distances, axis=-1).flatten())
                        min_point = min_//4
                        min_side = min_%4
                        v1 = bounding_box_[min_point]-mass
                        v2_ = distances[min_side]-mass
                        theta = np.arccos(v1@v2_ / (np.linalg.norm(v1)*np.linalg.norm(v2_)))
                        catet2 = v1 * np.sin(theta)
                        max_theta = np.arctan2(np.linalg.norm(catet2), np.linalg.norm(v2_))
                        turn_angle = (theta - max_theta, degr)[np.argmin([abs(theta - max_theta), abs(degr)])]
                    else:turn_angle=0
                else: 

                    turn_angle = (-5+ np.random.random()*10)/57.29 if self.turnProb >= np.random.uniform(0, 1) else 0
                    mass = (self.HWChannels["h"]//2,self.HWChannels["w"]//2)
            
                #cv2.line(, [0,0], np.array(image.shape[:2])-1, color=(255,255,255))
            
                rotation_matrix = cv2.getRotationMatrix2D(mass, turn_angle*57.29, 1)
            
                image_turned = cv2.warpAffine(image.copy(), M = rotation_matrix, dsize = image.shape[:2])
                mask_turned = target if isinstance(target, int) else cv2.warpAffine(target.copy(), M = rotation_matrix, dsize = image.shape[:2])
                
                augmentation = v2.Compose([
                    v2.ColorJitter(
                        brightness=(0.7, 1.5),
                        contrast=(0.7, 1.5),
                        saturation=(0.5, 1.7),
                        hue=(-0.3, 0.3),
                    ),
                    v2.RandomHorizontalFlip(p=0.4),
                    v2.RandomVerticalFlip(p=0.3),
                    v2.RandomApply(
                        [v2.GaussianBlur(kernel_size=5, sigma=(0.1, 0.5))],
                        p=0.5,
                    ),
                ])
                #augmentation.

                image = tv_tensors.Image(
                    torch.tensor(image_turned)
                        .permute(2, 0, 1)
                        .float() / 255
                )

                mask = mask_turned if isinstance(mask_turned, int) else tv_tensors.Mask(
                    torch.tensor(mask_turned).float() / 255
                )
                image, target = augmentation(image, mask) if not isinstance(mask_turned, int) else augmentation(image), mask
                # target = mask_turned if isinstance(mask_turned, int) else \
                # augmentation(torch.tensor(mask_turned).float() / 255)
            else: 
                image = torch.tensor(image).permute(2, 0, 1).float() / 255
                target = target if isinstance(target, int) else \
                    torch.tensor(target).float() / 255
            print(imageId_list.iloc[i])
            yield indices[i], {
                "path":imageId_list.iloc[i],
                "images":image,
                "labels":target
            }



In [9]:
from transformers import DataCollator

In [10]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorForImageClassification:
    ## SOME FIELDS
    device: str = "cuda"
    # processor: Any
    # decoder_start_token_id: int
    
    def __call__(self, features: List[Dict[str, Union[List[List[List[float]]], 
                                List[str], List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = []
        batch = dict()
        labels_batch = []
        masks_batch = [] #/ 255
        decoder_inputs = []
        #print(features)
        for feature in features:

            labels = torch.tensor(feature['labels'], dtype=torch.long) # Do I need to start from first element?
            images = torch.tensor(feature['images'], dtype=torch.float32) #/ 255
            
            input_features.append(images)
            labels_batch.append(labels)
        batch['labels'] = torch.hstack(labels_batch)#labels#
        batch['images'] = torch.stack(input_features)
        return batch

In [12]:
class UnetDownsampler(torch.nn.Module):
    def __init__(self, h, w, c, device="cuda"):
        super().__init__()
        self.device = device
        self.h = h
        self.w = w
        self.c = c

        self.input = torch.nn.Conv2d(
                in_channels = c,
                out_channels = 64,
                kernel_size = (7, 7),
                stride = 1,
                padding='same' 
            ).to(device=device, dtype=torch.bfloat16)

        self.activation = torch.nn.ReLU().to(device=device, dtype=torch.bfloat16)
        
        self.pool_input = torch.nn.MaxPool2d(
            kernel_size=(3, 3),
            stride=4,
        ).to(device=device, dtype=torch.bfloat16)
        self.conv_2 = [
            torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=64,
                            out_channels=64,
                            kernel_size=(3, 3),
                            stride=1,
                            padding='same'),
            torch.nn.Conv2d(in_channels=64,
                            out_channels=64,
                            kernel_size=(3, 3),
                            stride=1,
                            padding='same'),
            torch.nn.LayerNorm([64, h//4 , w//4 ]),
            self.activation).to(device=device, dtype=torch.bfloat16), 
            torch.nn.Sequential(    
            torch.nn.Conv2d(in_channels=64,
                                        out_channels=64,
                                        kernel_size=(3, 3),
                                        stride=1,
                padding='same'),
                        torch.nn.Conv2d(in_channels=64,
                                        out_channels=64,
                                        kernel_size=(3, 3),
                                        stride=1,
                padding='same'),
                        torch.nn.LayerNorm([64, h//4, w//4 ]),
                        self.activation,).to(device=device, dtype=torch.bfloat16)
                    ]          
        self.reduce_dim_0 = torch.nn.Conv2d(in_channels=64, out_channels=128, 
                                            kernel_size=(1,1), stride=2).to(device=device, dtype=torch.bfloat16)
        self.conv_3 = [torch.nn.Sequential(
                    torch.nn.Conv2d(in_channels=64,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                        padding='same'),
                    torch.nn.MaxPool2d(
                    kernel_size=(2, 2),
                    stride=2,
                    ),
                    torch.nn.Conv2d(in_channels=128,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                     padding='same'),
                    torch.nn.LayerNorm([128, h//8, w//8]),
                    self.activation,              
                ).to(device=device, dtype=torch.bfloat16), torch.nn.Sequential(
                    torch.nn.Conv2d(in_channels=128,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                        padding='same'),
                    torch.nn.Conv2d(in_channels=128,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                         padding='same'),
                    torch.nn.LayerNorm([128, h//8, w//8]),
                    self.activation,              
                ).to(device=device, dtype=torch.bfloat16),
                ]
        self.reduce_dim_1 = torch.nn.Conv2d(in_channels=128, out_channels=256, 
                                                    kernel_size=(1,1), stride=2,
                                                    ).to(device=device, dtype=torch.bfloat16)
        self.conv_4 = [torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=128,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                torch.nn.MaxPool2d(
                    kernel_size=(2, 2),
                    stride=2,
                    ),
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.LayerNorm([256, h//16, w//16]),
                            self.activation,              
                        ).to(device=device, dtype=torch.bfloat16), torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.LayerNorm([256, h//16, w//16]),
                            self.activation,              
                        ).to(device=device, dtype=torch.bfloat16),
                        ]
        self.reduce_dim_2 = torch.nn.Conv2d(in_channels=256, out_channels=512, 
                                                            kernel_size=(1,1), stride=2).to(device=device, dtype=torch.bfloat16)
        self.conv_5 = [torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                torch.nn.MaxPool2d(
                    kernel_size=(2, 2),
                    stride=2,
                    ),
                            torch.nn.Conv2d(in_channels=512,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.LayerNorm([512, h//32, w//32]),
                            self.activation,              
                        ).to(device=device, dtype=torch.bfloat16), torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=512,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.Conv2d(in_channels=512,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.LayerNorm([512, h//32, w//32]),
                            self.activation,              
                        ).to(device=device, dtype=torch.bfloat16),
                        ]

    def forward(self, images):
        ## Create implementation
        res_0 = self.input(images)
        #print(res_0.shape)
        res_0 = self.pool_input(self.activation(res_0))
        print("1")
        x_0 = self.conv_2[0](res_0) 
        #print(res_0.device, x_0.device, torch.nn.functional.pad(res_0, (0,0, 0,0, 0, x_0.shape[0]- res_0.shape[0]),mode="constant",value = 0).to(device=self.device , dtype=torch.bfloat16).device)
        x_0 += torch.nn.functional.pad(res_0, (0,0, 0,0, 0, x_0.shape[0]- res_0.shape[0]),mode="constant",value = 0).to(device=self.device , dtype=torch.bfloat16)
        
        x_1 = self.conv_2[1](x_0) 
        x_1 += torch.nn.functional.pad(res_0, (0,0, 0,0, 0, x_1.shape[0]- x_0.shape[0]), mode="constant", value = 0).to(device=self.device , dtype=torch.bfloat16)+ x_1
        print("2")
        res_1 = self.reduce_dim_0(x_1)
        #print(res_1.shape)
        x_2 = self.conv_3[0](x_1)
        #print(x_2.shape, x_1.shape)
        x_2+=res_1
        x_3 = self.conv_3[1](x_2)
        x_3 += torch.nn.functional.pad(res_1, (0,0, 0,0, 0, x_3.shape[0]- x_2.shape[0]), mode="constant", value = 0).to(device=self.device , dtype=torch.bfloat16)+ x_3
        print("3")
        res_2 = self.reduce_dim_1(x_3)
        x_4 = self.conv_4[0](x_3)+res_2
        x_5 = self.conv_4[1](x_4)
        x_5 += torch.nn.functional.pad(res_2, (0,0, 0,0, 0, x_5.shape[0]- x_4.shape[0]), mode="constant", value = 0).to(device=self.device , dtype=torch.bfloat16)+ x_5
        print("4")
        res_3 = self.reduce_dim_2(x_5)
        x_6 = self.conv_5[0](x_5)+res_3
        x_7 = self.conv_5[1](x_6)
        x_7 += torch.nn.functional.pad(res_3, (0,0, 0,0, 0, x_7.shape[0]- x_6.shape[0]), mode="constant", value = 0).to(device=self.device , dtype=torch.bfloat16)+ x_7

        return x_7




In [13]:
class Classification_Model(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.loss_fn = torch.nn.CrossEntropyLoss()

    def forward(self, images, labels=None):
        logits = self.model(images)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

In [14]:
for name, p in model.named_parameters():
    print(name, p.device)

NameError: name 'model' is not defined

In [15]:
full_model

NameError: name 'full_model' is not defined

In [16]:
model

NameError: name 'model' is not defined

In [17]:
torch.nn.Sequential(model, 
                                 torch.nn.Flatten(),
                                 torch.nn.Linear(512*(model.h//32)*(model.w//32),
                                out_features=1000),
                                torch.nn.ReLU(),
                                torch.nn.Linear(1000, 2),
                                torch.nn.Softmax()).to(device="cuda", dtype=torch.bfloat16)

NameError: name 'model' is not defined

In [18]:
model = UnetDownsampler(c = 3, h = 768, w = 768)



full_model = Classification_Model(torch.nn.Sequential(model, 
                                 torch.nn.Flatten(),
                                 torch.nn.Linear(512*(model.h//32)*(model.w//32),
                                out_features=1000),
                                torch.nn.ReLU(),
                                torch.nn.Linear(1000, 2),
                                torch.nn.Softmax()).to(device="cuda", dtype=torch.bfloat16))

In [19]:
full_model = full_model.to(device="cuda",dtype=torch.bfloat16)

In [ ]:
from transformers import TrainingArguments

loss = torch.nn.BCELoss()
n_classes = 2
metric = multiclass_f1_score
print("eval")
def compute_metrics(pred):
    with torch.no_grad():
        pred_ids = pred.predictions
        label_ids = pred.label_ids

         # handle tuple
        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]

        # logits → token ids
        if pred_ids.ndim == 3:
            pred_ids = pred_ids.argmax(axis=-1)

        score = 100 * metric.compute(input=pred_ids, target=label_ids, num_classes=2)
    print('label: ', label_ids)
    print('predictions', pred_ids)
    return {"f1_score": score}

def compute_loss_func(outputs, labels, num_items_in_batch):
    return loss(outputs.argmax(-1), labels)

training_args = TrainingArguments(
    output_dir="./model_classification_dir",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-6,
    warmup_ratio=0.2,
    #max_grad_norm=1,
    lr_scheduler_type="cosine",
    #label_smoothing_factor=0.00001,
    #num_train_epochs=4,
    max_steps=50,
    #weight_decay=0.05,
    bf16=True,
    fp16=False,
    save_strategy="steps",
    save_total_limit=10,
    greater_is_better=True,
    metric_for_best_model="f1_score",
    eval_strategy="steps",
    per_device_eval_batch_size=32,
    eval_steps=50,
    save_steps=50,
    report_to=["tensorboard"],
    push_to_hub=False,
    eval_on_start = True,
    eval_accumulation_steps=1,
    load_best_model_at_end=True,
)

eval


In [35]:
from transformers import Trainer

ds = BackBoneDataset(train_segmentations, False)


In [36]:
train_ds

IterableDataset({
    features: ['path', 'images', 'labels'],
    num_shards: 1
})

In [37]:
train_ds = ds.as_streaming_dataset(split="train")
test_ds = ds.as_streaming_dataset(split="test")
trainer = Trainer(
    args=training_args,
    model=full_model.to(device="cuda"),
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DataCollatorForImageClassification(),
    
    compute_metrics=compute_metrics,
    #compute_loss_func = compute_loss_func,

)

In [38]:
full_model

Classification_Model(
  (model): Sequential(
    (0): UnetDownsampler(
      (input): Conv2d(3, 64, kernel_size=(7, 7), stride=(1, 1), padding=same)
      (activation): ReLU()
      (pool_input): MaxPool2d(kernel_size=(3, 3), stride=4, padding=0, dilation=1, ceil_mode=False)
      (reduce_dim_0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2))
      (reduce_dim_1): Conv2d(128, 256, kernel_size=(1, 1), stride=(2, 2))
      (reduce_dim_2): Conv2d(256, 512, kernel_size=(1, 1), stride=(2, 2))
    )
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=294912, out_features=1000, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1000, out_features=2, bias=True)
    (5): Softmax(dim=None)
  )
  (loss_fn): CrossEntropyLoss()
)

In [39]:
with torch.no_grad():
    a = full_model(torch.randn(32, 3, 768, 768).to(device="cuda", dtype=torch.bfloat16))

1
2
3
4


/home/fakeegor/my_projects/my_ASR_project/tuning_proj/training_venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1775: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


In [40]:
trainer.train()


00003e153.jpg

000155de5.jpg

000194a2d.jpg

000194a2d.jpg

0001b1832.jpg

00021ddc3.jpg

00021ddc3.jpg

000303d4d.jpg

00031f145.jpg

00057a50d.jpg

000592296.jpg

0005d6d95.jpg

0006c52e8.jpg

000811bb6.jpg

0008d6216.jpg

000913daf.jpg

000952d51.jpg

000a4d4fd.jpg

000baef0c.jpg

000bd9ac4.jpg

000d26c17.jpg

000e37fc6.jpg

000f1f959.jpg

000f7e728.jpg

0010e88ce.jpg

0010eefb1.jpg

00113a75c.jpg

00113a75c.jpg

00113a75c.jpg

0011a9ccb.jpg

0011c31b7.jpg

00122061c.jpg

001234638.jpg

0013b2222.jpg

00140e597.jpg

00160335a.jpg

00167446e.jpg

0016c1d02.jpg

001739657.jpg

001768e84.jpg

00189cc36.jpg

0018e5d6c.jpg

0018f7728.jpg

001948368.jpg

001a7cba8.jpg

001ac9fa5.jpg

001aee007.jpg

001aee007.jpg

001eda01f.jpg

001f04ca3.jpg

002237256.jpg

0023b39c3.jpg

00240e745.jpg

0026f7ea5.jpg

002868a5c.jpg

00286e8b9.jpg

0029673da.jpg

0029a4e84.jpg

002a4ab34.jpg

002aae251.jpg

002aef0e2.jpg

002c872c1.jpg

002d56570.jpg

002deeb16.jpg

002e42551.jpg

002e8a977.jpg

002eba124

KeyboardInterrupt: 

In [397]:
->features
<-batch

SyntaxError: invalid syntax (1497795642.py, line 1)

In [ ]:
ds = BackBoneDataset(train_segmentations, False)
### COMPLETED
### 1.Test of download and prepare with streaming, using 1. no mask 2.mask
### 2. Try to externally shuffle the entries in the dataset splits for the data to check if the randomness works(of the streaming order)
### 3. check if the augmentations are random enough(check whether pytorch and np seed is correct)
### 4. Write further stuff and prove that it loads correctly for next steps

### TODO
### 4.6 Create DataCollator
### 4.8 Create Model 
### 5 Create Trainer object and compute loss func for classification

### 5. Create Model class and Callbacks for randomization(from step 2), create collator for putting everything on gpu
### 6. Create train test loops for classification
ds = BackBoneDataset(train_segmentations[~train_segmentations.EncodedPixels.isna()], True)

In [29]:
np.random.seed(47)
ds = BackBoneDataset(train_segmentations[~train_segmentations.EncodedPixels.isna()], True)
np.random.seed(47)
ds1 = BackBoneDataset(train_segmentations[~train_segmentations.EncodedPixels.isna()], True)

np.random.shuffle(ds1.training_paths)
np.random.shuffle(ds1.testing_paths)

In [20]:
ds1.testing_paths

array([     2,      3,      5, ..., 131044, 131061, 131067],
      shape=(36806,))

In [81]:
train_segmentations[~train_segmentations.EncodedPixels.isna()]

,ImageId,EncodedPixels
2,000155de5.jpg,264661 17 265429 33 266197 33 266965 33 267733...
3,000194a2d.jpg,360486 1 361252 4 362019 5 362785 8 363552 10 ...
4,000194a2d.jpg,51834 9 52602 9 53370 9 54138 9 54906 9 55674 ...
5,000194a2d.jpg,198320 10 199088 10 199856 10 200624 10 201392...
6,000194a2d.jpg,55683 1 56451 1 57219 1 57987 1 58755 1 59523 ...
...,...,...
231694,fff909871.jpg,441057 2 441825 4 442592 7 443360 9 444128 11 ...
231709,fffcb6464.jpg,222782 4 223550 10 224318 14 225086 14 225854 ...
231712,fffd924fb.jpg,110011 5 110779 5 111547 5 112315 5 113083 5 1...
231713,fffd924fb.jpg,198440 8 199208 8 199976 8 200744 8 201512 8 2...


In [30]:
train_ds = ds.as_streaming_dataset(split="train")
test_ds = ds.as_streaming_dataset(split="test")

train_ds__ = ds1.as_streaming_dataset(split="train")
test_ds__ = ds1.as_streaming_dataset(split="test")

In [31]:
for example in train_ds:
    res = example
    break

for example in train_ds__:
    res1 = example
    break

In [35]:
from torchvision.transforms.functional import to_pil_image
img = res1["image"][0].copy()
#img*=example["image"][1]
img_tensor = torch.Tensor(img)  # e.g. shape [C, H, W]
img_pil = to_pil_image(img_tensor)

img_pil.show()

In [92]:
example['image'][1]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(768, 768), dtype=float32)

In [88]:
example['image'][0]

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]],
      shape=(3, 768, 768), dtype=float32)

In [20]:
v2

<module 'torchvision.transforms.v2' from '/Users/ruslanomchenko/Documents/new_code_projects/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/__init__.py'>

In [66]:
example

{'path': 'bdf204e86.jpg',
 'image': array([[[0.54650086, 0.48763362, 0.4640866 , ..., 0.6488867 ,
          0.66066015, 0.66654694],
         [0.51706713, 0.4640866 , 0.43465286, ..., 0.63711333,
          0.6488867 , 0.6547734 ],
         [0.46997336, 0.43465286, 0.39933252, ..., 0.619453  ,
          0.63122654, 0.64299995],
         ...,
         [0.6309141 , 0.6309141 , 0.6309141 , ..., 0.5014059 ,
          0.5426129 , 0.56615996],
         [0.64268756, 0.6309141 , 0.6250274 , ..., 0.4837457 ,
          0.5072926 , 0.5249528 ],
         [0.66034776, 0.6368008 , 0.6309141 , ..., 0.46019867,
          0.47197223, 0.47785896]],
 
        [[0.58350956, 0.5246422 , 0.5010953 , ..., 0.6766402 ,
          0.6884137 , 0.6943004 ],
         [0.5540758 , 0.5010953 , 0.47166163, ..., 0.66486675,
          0.6766402 , 0.6825269 ],
         [0.5069821 , 0.47166163, 0.43634126, ..., 0.6472065 ,
          0.65897995, 0.6707534 ],
         ...,
         [0.6355418 , 0.6355418 , 0.6355418 , ..., 0